### Pré-Processamento

In [10]:
import json

f = open("dataset_articles.json")
papers = json.load(f)
len(papers)

papers_f = []
for p in papers:
    if p["abstract"] and len(p["abstract"]) > 150 and p["keywords"] and len(p["keywords"]) > 1:
        papers_f.append(p)

len(papers_f)

473

### Calculate abstract pairs

In [23]:
from itertools import combinations
import re

def processa_keywords(s):
    s = s.lower().strip()
    keywords = re.split(r"[,;]",s)
    return [k.strip() for k in keywords] 

abstract_pairs = []
for p1, p2 in combinations(papers_f, 2):
    
    p2["abstract"]
    k1 = processa_keywords(p1["keywords"])
    k2 = processa_keywords(p2["keywords"])
    score = len(set(k1) & set(k2))
    abstract_pairs.append((p1["abstract"], p2["abstract"], score))
    if score == 4:
        print(k1,k2)
        print(p1["abstract"], p2["abstract"], sep="\n\n")

print(abstract_pairs[0])

['doença de still do adulto', 'febre', 'artralgias', 'exantema', 'ferritina'] ['doença still', 'ferritina', 'febre', 'artralgias', 'exantema']
A Doença de Still do Adulto (DSA) é uma doença inflamatória sistémica, rara, de etiologia desconhecida, cujas principais manifestações incluem febre elevada, exantema maculopapular evanescente, artralgias/artrite e odinofagia persistente. O diagnóstico é clínico e implica a exclusão de outras patologias infeciosas, autoimunes e neoplásicas. Os autores apresentam o caso clínico de doente de 22 anos, previamente saudável, com quadro com cerca de 5 meses de evolução, caraterizado por poliartralgias simétricas de ritmo inflamatório, com incapacidade funcional marcada, associadas a febre vespertina, exantema evanescente, astenia, anorexia e perda ponderal. Ao exame físico apresentava mucosas descoradas, sinovite das articulações metacarpofalangicas e interfalangicas proximais, e múltiplas adenopatias infra e pericentimétricas, não dolorosas, móveis, 

In [17]:
from collections import Counter
Counter([score for _, _, score in abstract_pairs])

Counter({0: 110992, 1: 601, 2: 32, 3: 2, 4: 1})

### Balance abstract pairs

In [24]:
from sklearn.utils import resample

majority_class = [p for p in abstract_pairs if p[2] == 0]
minority_class = [p for p in abstract_pairs if p[2] != 0]

majority_balanced_pairs  = resample(
                            majority_class,
                            replace = False,
                            n_samples= len(minority_class),
                            random_state=42,
)

balanced_pairs = majority_balanced_pairs + minority_class
Counter([score for _, _, score in balanced_pairs])

Counter({0: 636, 1: 601, 2: 32, 3: 2, 4: 1})

### Normalize scores

In [38]:
def normalize_scores(score):
    if score == 0:
        return 0
    if score == 1:
        return 0.5
    if score == 2:
        return 0.75
    if score >= 3:
        return 0.85
    
balanced_pairs_norm = [(a1,a2, normalize_scores(score)) for a1, a2, score in balanced_pairs]
balanced_pairs_norm[637]

('As vasculites caracterizam-se por inflamação e necrose de vasos sanguíneos, frequentemente de causa autoimune, porém podem ser secundárias, por exemplo, a infeções. Relata-se um caso de vasculite secundária com atingimento cutâneo, articular e hematológico por infeção pulmonar por Entamoeba histolytica, num homem de 36 anos com história prévia de viagem à América do Sul.\u202f',
 'Descrevemos o caso de uma doente de 79 anos, natural de Lisboa, sem história epidemiológica relevante, com antecedentes conhecidos de quistos hepáticos, dispepsia e hipertensão arterial. Recorreu ao Serviço de Urgência (SU) por febre e desconforto abdominal sem outras queixas de orgão ou sistema; analiticamente, salientava-se elevação de parâmetros inflamatórios; transaminases e parâmetros de colestase sem alterações; realizou hemoculturas e iniciou terapêutica empírica com amoxicilina/clavulanato com má evolução; tomografia computadorizada (TC) do abdómen revelou quisto hepático complicado; o estudo etioló

### Generate train test split

In [43]:
from sklearn.model_selection import train_test_split

scores = [score for _, _ , score in balanced_pairs_norm]

train_data, test_data = train_test_split(
    balanced_pairs_norm,
    test_size=0.2,
    random_state=42,
    stratify = scores
)


print(Counter([score for _, _, score in train_data]))
print(Counter([score for _, _, score in test_data]))

Counter({0: 508, 0.5: 481, 0.75: 26, 0.85: 2})
Counter({0: 128, 0.5: 120, 0.75: 6, 0.85: 1})


In [51]:
from datasets import Dataset

train_dataset = Dataset.from_list([{"abstract": a1, "abstract2": a2, "score": score} for a1, a2, score in train_data])
test_dataset = Dataset.from_list([{"abstract1": a1, "abstract2": a2, "score": score} for a1, a2, score in test_data])

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['abstract', 'abstract2', 'score'],
    num_rows: 1017
})
Dataset({
    features: ['abstract1', 'abstract2', 'score'],
    num_rows: 255
})


### Model Training

In [54]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.sentence_transformer.losses import CosineSimilarityLoss

model = SentenceTransformer("neuralmind/bert-base-portuguese-cased")
loss = CosineSimilarityLoss(model)

In [48]:
from sentence_transformers import SentenceTransformerTrainingArguments

args = SentenceTransformerTrainingArguments(
    # Required parameter:
    output_dir="models/bi_encoder_medicina",
    # Optional training parameters:
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    warmup_steps=0.1,
    fp16=True,  # Set to False if you get an error that your GPU can't run on FP16
    bf16=False,  # Set to True if you have a GPU that supports BF16
    # Optional tracking/debugging parameters:
    eval_strategy="epoch",
    eval_steps=100,
    save_strategy="epoch",
    save_steps=100,
    save_total_limit=2,
    logging_steps=100,
)

In [52]:
from sentence_transformers.sentence_transformer.evaluation import EmbeddingSimilarityEvaluator, SimilarityFunction

# Initialize the evaluator
dev_evaluator = EmbeddingSimilarityEvaluator(
    sentences1=test_dataset["abstract1"],
    sentences2=test_dataset["abstract2"],
    scores=test_dataset["score"],
    main_similarity=SimilarityFunction.COSINE
)
# You can run evaluation like so:
# results = dev_evaluator(model)

In [55]:
from sentence_transformers import SentenceTransformerTrainer
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    loss=loss,
    evaluator=dev_evaluator,
)


In [56]:
trainer.train()

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

### Inference

In [ ]:
model = SentenceTransformer("lfcc/medlink-bi-encoder")

In [62]:
query = """S/
Identidicação: género feminino, 24 anos.
 
AP:
# sem antecedentes conhecidos.
 
MH:
# contraceptivo oral
 
HDA: Recorre ao serviço de urgência por nódulos violáceos dolorosos na região pré-tibial bilateral, que estenderam a toda a perna e coxa, com 1 mês de evolução. Três semanas antes do início do quadro, realizou a 2ª dose da vacina contra SARS-CoV-2 Comirnaty® (Pfizer-BioNTech), a mesma que realizou um mês antes. Sem outras queixas, nomeadamente sugestivas de patologia auto-imune ou síndrome onstitucional.
 
O/
Exame físico- apirética, sem alterações da orofaringe ou adenopatias, observando-se lesões nodulares em ambas as coxas e pernas, de cor violácea, eritematosas, com 2 cm de maior eixo, dolorosas à palpação.
 
A/
# Analiticamente- leucocitose de 12,6 x 109/L com neutrofilia, proteína C reactiva 210 mg/L e velocidade de sedimentação 42 mm/h.
# Anticorpo anti-estreptolisina O ne-gativo, serologias para sífilis e vírus Ebstein-Barr, hepatite B e C e VIH negativas. Painel imunológico, com pesquisa de anticorpos anti-nucleares (ANA), anticorpos anti-double--stranded DNA (dsDNA), anticorpos itoplasmáticos anti--neutrófilos (ANCA), complemento, factor reumatóide e anticorpos anti-péptido citrulinado cíclico, negativos.
# Marcadores de lesão hepática e função renal sem alterações.
# Hemoculturas negativas.
# TC TAP sem alterações.
 
Assim, assumido eritema nodoso secundário à administração da vacina contra SARS-CoV-2.
Iniciada prednisolona 60 mg/dia.
 
Evolução:
# Uma semana depois houve melhoria clínica significativa e diminuição acentuada dos parâmetros inflamatórios, pelo que se iniciou redução gradual de corticoterapia, com resolução completa do quadro ao fim de duas semanas.
# Passados seis meses, a doente teve doença ligeira por SARS-CoV-2 e uma semana depois surgiram lesões com características idênticas às do episódio anterior. Iniciou novo ciclo de corticoterapia, com resolução do quadro. Pela exuberância das lesões, a doente foi aconselhada a não repetir imunização com vacina de mRNA contra SARS-CoV-2.
# Até à data, não voltou ter infecção por SARS-CoV-2 nem teve ressurgimento das lesões cutâneas."""

In [70]:
abstracts = [p["abstract"] for p in papers_f]
keywords = [p["keywords"] for p in papers_f]

abstract_embeddings = model.encode(abstracts)


In [72]:
from sentence_transformers import util
import torch
query_embeddings = model.encode(query)

cosine_scores = util.pytorch_cos_sim(query_embeddings, abstract_embeddings)
#print(cosine_scores)

ir_res = torch.topk(cosine_scores, k=15)
#print(ir_res)

for cos, idx in zip(ir_res.values[0],ir_res.indices[0]):
    print("score: ", cos)
    print("Abstract: ", abstracts[idx])
    print("Keywords: ", keywords[idx])
    print("-"*20)
    


score:  tensor(0.6316)
Abstract:  Resumo A síndrome de Evans é rara, existindo poucos casos descritos de associação a vacina mRNA SARS-CoV-2, nenhum deles em Portugal. Doente do sexo masculino de dezoito anos, admitido pordor abdominal, icterícia, urina escura e cansaço. Na avaliação apresentava anemia hemolítica autoimune por anticorpos quentes, tendo desenvolvido posteriormente trombocitopenia autoimune. Dos antecedentes destacava-se vasculite IgA aos vinte meses e administração de vacina mRNA SARS-CoV-2 nove dias antes da admissão. Foram excluídas infeções virais, doenças autoimunes/linfoproliferativas, imunodeficiências, fármacos e trombofilias. Apesar de não ter apresentado resposta inicial a corticoterapia e imunoglobulina, verificou-se evolução favorável após a introdução de rituximab e ciclofosfamida. Dada a relação temporal e a exclusão de outras causas, foiefetuado o diagnóstico de síndrome de Evans provavelmente desencadeado por vacina mRNA SARS-CoV-2. É o primeiro caso desc